In [1]:
import sys
from pathlib import Path
# Add parent directory to path to import from src/
sys.path.append(str(Path(__file__).parent.parent) if '__file__' in globals() else str(Path.cwd().parent))

from sionna.rt import load_scene, RadioMapSolver, Camera, transform_mesh, MeshRadioMap, PlanarRadioMap
from src.utils import load_building_positions, get_antenna_positions, get_scene_bounds
from src.base_station import set_tx_antenna_array, add_base_station
from src.user_equipment import set_rx_antenna_array

jitc_llvm_init(): LLVM API initialization failed ..


In [2]:
# Define parameters
NUM_DEPLOYMENT_BUILDINGS = 3
ANTENNA_HEIGHT_OFFSET = 10.0
SHIFT_FACTOR = 0.25
SCENE_CENTER = [0.0, 0.0]
ELEVATION = True
scene_dir = Path("../scenes/boston_1/")
scene_xml_path = scene_dir / "scene.xml"
building_positions_path = scene_dir / "buildings.json"

In [3]:
# setup scene and measurement surface
scene = load_scene(scene_xml_path)
bbox_min, bbox_max = get_scene_bounds(scene)

if ELEVATION:
    measurement_surface = scene.objects["ground"].clone(as_mesh=True)
    transform_mesh(measurement_surface,
                translation=[0,0,1.5])
else:
    measurement_surface = None

building_positions = load_building_positions(building_positions_path)

antenna_information = get_antenna_positions(
    building_positions, 
    scene_center=SCENE_CENTER,
    antenna_height_offset=ANTENNA_HEIGHT_OFFSET,
    shift_factor=SHIFT_FACTOR,
    num_deployment_buildings=NUM_DEPLOYMENT_BUILDINGS)

FileNotFoundError: [Errno 2] No such file or directory: '../scenes/boston_1/scene.xml'

In [4]:
# Position camera at a good viewing distance from the scene center at look at the scene center
camera = Camera(position=bbox_max, look_at=[0, 0, 0])
scene.render(camera=camera);

NameError: name 'bbox_max' is not defined

In [5]:
# set tx and rx specs and add base stations
set_tx_antenna_array(
    scene,
    num_rows=8,
    num_cols=8,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross")

set_rx_antenna_array(
    scene,
    num_rows=2,
    num_cols=2,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="hw_dipole",
    polarization="VH"
)

for i, current_antenna_information in enumerate(antenna_information):
    bs_name = f"BS_{i}"
    _, antenna_position = current_antenna_information
    print(f"Adding base station {bs_name} at position {antenna_position}")
    add_base_station(
        scene,
        bs_name,
        position=antenna_position,
        num_sectors=6,
        mechanical_tilt=10.0,
        azimuth_offset=0.0,
        tx_power_dbm=43.0,
        display_radius=15.0
    )

NameError: name 'scene' is not defined

In [6]:
scene.preview();

NameError: name 'scene' is not defined

In [7]:
rm_solver = RadioMapSolver()
rm = rm_solver(scene,
               measurement_surface=measurement_surface,
               diffuse_reflection=True,
               diffraction=True,
               edge_diffraction=True,
               max_depth=5,           # Maximum number of ray scene interactions
               samples_per_tx=10**6) # If you increase: less noise, but more memory required

NameError: name 'scene' is not defined

In [8]:
scene.objects

{'no-name-1': <sionna.rt.scene_object.SceneObject at 0x73d6a3537d50>,
 'no-name-2': <sionna.rt.scene_object.SceneObject at 0x73d6a36c9d50>,
 'ground': <sionna.rt.scene_object.SceneObject at 0x73d740196390>}

In [ ]:
if isinstance(rm, MeshRadioMap):
    scene.preview(radio_map=rm, rm_db_scale=True, rm_metric="path_gain");

In [ ]:
if isinstance(rm, PlanarRadioMap):
    rm.show(metric="path_gain", tx=0);